# Alzheimer's Early Warning System - Modality-Specific AI Models

This notebook trains separate AI models for each modality:
- **Speech**: Audio features (pause, repetition, pitch, transcription)
- **Text**: Writing features (complexity, vocabulary, coherence)
- **Behavior**: Time-series features (trend, seasonality, anomalies)

Then combines them with a **Multimodal Fusion Model**.


## 1. Google Colab Setup (Run this first if using Colab)


In [ ]:
# Check if running in Google Colab
IN_COLAB = False
try:
    import google.colab
    IN_COLAB = True
    print("🔵 Running in Google Colab")
except ImportError:
    print("💻 Running locally")

if IN_COLAB:
    # Mount Google Drive (optional - for persistent storage)
    from google.colab import drive
    try:
        drive.mount('/content/drive')
        print("✅ Google Drive mounted")
    except Exception as e:
        print(f"ℹ️ Drive mount skipped: {e}")
    
    # Comprehensive dependency installation
    print("📦 Installing all required packages...")
    import subprocess
    import sys
    
    def install_package(package_name, description=""):
        """Install a package and report status."""
        try:
            result = subprocess.run(
                [sys.executable, '-m', 'pip', 'install', '-q', package_name],
                capture_output=True,
                text=True,
                timeout=300
            )
            if result.returncode == 0:
                print(f"   ✅ {package_name} {description}")
                return True
            else:
                print(f"   ⚠️ {package_name} installation had issues (may already be installed)")
                return False
        except Exception as e:
            print(f"   ⚠️ {package_name} installation failed: {str(e)[:50]}")
            return False
    
    # Core ML libraries (required)
    print("\n🔧 Core ML Libraries:")
    install_package("numpy", "(>=1.21.0)")
    install_package("pandas", "(>=1.3.0)")
    install_package("scikit-learn", "(>=1.0.0)")
    install_package("xgboost", "(>=1.5.0)")
    install_package("lightgbm", "(>=3.3.0)")
    install_package("joblib", "(>=1.1.0)")
    
    # Optional ML libraries
    print("\n🔧 Optional ML Libraries:")
    install_package("catboost", "(optional)")
    
    # Audio processing (for speech features)
    print("\n🔧 Audio Processing Libraries:")
    install_package("librosa", "(for audio analysis)")
    install_package("soundfile", "(for audio I/O)")
    install_package("openai-whisper", "(for transcription)")
    install_package("pydub", "(for audio conversion)")
    
    # NLP libraries (for text features)
    print("\n🔧 NLP Libraries:")
    install_package("spacy", "(for text analysis)")
    install_package("transformers", "(for advanced NLP)")
    install_package("torch", "(for transformers)")
    
    # Download spaCy model
    try:
        subprocess.run([sys.executable, '-m', 'spacy', 'download', 'en_core_web_sm'], 
                      capture_output=True, timeout=120)
        print("   ✅ spaCy English model downloaded")
    except Exception as e:
        print(f"   ⚠️ spaCy model download failed: {str(e)[:50]}")
        print("   💡 Run manually: python -m spacy download en_core_web_sm")
    
    # Time-series libraries (for behavior features)
    print("\n🔧 Time-Series Libraries:")
    install_package("prophet", "(for trend analysis)")
    install_package("pystan", "(Prophet dependency)")
    
    # Visualization libraries
    print("\n🔧 Visualization Libraries:")
    install_package("matplotlib", "(for plotting)")
    install_package("seaborn", "(for statistical plots)")
    
    # Additional utilities
    print("\n🔧 Additional Utilities:")
    install_package("tqdm", "(for progress bars)")
    install_package("scipy", "(for scientific computing)")
    
    print("\n✅ Package installation complete!")
    print("💡 Note: Some packages may take time to download models on first use.")
    
    # Verify key packages
    print("\n🔍 Verifying key packages...")
    key_packages = {
        'numpy': 'numpy',
        'pandas': 'pandas',
        'sklearn': 'scikit-learn',
        'xgboost': 'xgboost',
        'lightgbm': 'lightgbm',
        'librosa': 'librosa',
        'spacy': 'spacy',
        'transformers': 'transformers',
        'prophet': 'prophet',
        'whisper': 'openai-whisper'
    }
    
    available = {}
    for import_name, package_name in key_packages.items():
        try:
            __import__(import_name)
            available[package_name] = True
        except ImportError:
            available[package_name] = False
    
    print("   Status:")
    for pkg, status in available.items():
        status_icon = "✅" if status else "❌"
        print(f"   {status_icon} {pkg}")
    
    missing = [pkg for pkg, status in available.items() if not status]
    if missing:
        print(f"\n⚠️ Missing packages: {', '.join(missing)}")
        print("   These may be optional or may need manual installation.")
    
    # Note about ffmpeg (for audio conversion)
    print("\n💡 Note about ffmpeg:")
    print("   Colab may have ffmpeg pre-installed. If audio conversion fails,")
    print("   you can install it with: !apt-get install -y ffmpeg")
    
    # Clone repository or set up project structure
    import os
    if not os.path.exists('/content/Alzhemiers'):
        print("📥 Setting up project structure...")
        os.makedirs('/content/Alzhemiers', exist_ok=True)
        os.chdir('/content/Alzhemiers')
        # Create necessary directories
        os.makedirs('data/processed', exist_ok=True)
        os.makedirs('models/saved/modality_models', exist_ok=True)
        os.makedirs('scripts', exist_ok=True)
        os.makedirs('models', exist_ok=True)
        print("✅ Project directories created")
    else:
        os.chdir('/content/Alzhemiers')
        print("✅ Using existing project directory")
    
    # For Colab, we'll need to upload the required Python files
    print("\n📋 Next steps:")
    print("   1. Upload your feature CSV files (speech_features.csv, text_features.csv, behavior_features.csv, labels.csv)")
    print("   2. Upload models/train_tabular.py if you have it")
    print("   3. Or use the inline model definitions (see next cell)")
    
    PROJECT_ROOT = '/content/Alzhemiers'
else:
    # Local setup
    PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), '..'))
    if not os.path.exists(PROJECT_ROOT):
        PROJECT_ROOT = os.getcwd()

print(f"📁 Project root: {PROJECT_ROOT}")
os.chdir(PROJECT_ROOT)


## 2. Imports and Model Definitions


In [ ]:
import os
import sys
import numpy as np
import pandas as pd
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')

# Add project root to path
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

# ML imports
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.metrics import (
    accuracy_score, roc_auc_score, precision_score, recall_score, f1_score,
    classification_report, confusion_matrix
)
from sklearn.preprocessing import StandardScaler
import joblib

# Models
import xgboost as xgb
import lightgbm as lgb
from sklearn.ensemble import (
    RandomForestClassifier, ExtraTreesClassifier, GradientBoostingClassifier
)
from sklearn.linear_model import LogisticRegression
from sklearn.neural_network import MLPClassifier

# Try to import CatBoost (optional)
try:
    from catboost import CatBoostClassifier
    CATBOOST_AVAILABLE = True
except ImportError:
    CATBOOST_AVAILABLE = False
    print("ℹ️ CatBoost not available (optional)")

# Try to import from existing module, otherwise define inline
try:
    from models.train_tabular import build_models, train_models
    print("✅ Using existing model builder from models/train_tabular.py")
except ImportError:
    print("⚠️ models/train_tabular.py not found. Defining models inline...")
    
    def build_models(random_state: int = 42):
        """Build model dictionary."""
        models = {
            'XGBoost': xgb.XGBClassifier(
                n_estimators=300, max_depth=6, learning_rate=0.05,
                subsample=0.8, colsample_bytree=0.8, reg_alpha=0.5, reg_lambda=1.5,
                random_state=random_state, tree_method='hist', eval_metric='logloss',
                verbosity=0, n_jobs=-1
            ),
            'LightGBM': lgb.LGBMClassifier(
                n_estimators=300, max_depth=-1, num_leaves=31, learning_rate=0.05,
                subsample=0.8, colsample_bytree=0.8, reg_alpha=0.5, reg_lambda=1.5,
                min_child_samples=20, random_state=random_state, verbose=-1, n_jobs=-1
            ),
            'RandomForest': RandomForestClassifier(
                n_estimators=300, max_depth=15, min_samples_split=5, min_samples_leaf=2,
                max_features='sqrt', max_samples=0.8, n_jobs=-1, random_state=random_state, oob_score=True
            ),
            'ExtraTrees': ExtraTreesClassifier(
                n_estimators=300, max_depth=15, min_samples_split=5, min_samples_leaf=2,
                max_features='sqrt', max_samples=0.8, n_jobs=-1, random_state=random_state
            ),
            'LogisticRegression': LogisticRegression(
                max_iter=2000, solver='lbfgs', C=1.0, class_weight='balanced',
                random_state=random_state, multi_class='ovr', n_jobs=-1
            ),
            'MLP': MLPClassifier(
                hidden_layer_sizes=(128, 64), activation='relu', solver='adam', alpha=0.01,
                learning_rate='adaptive', max_iter=500, early_stopping=True,
                validation_fraction=0.1, n_iter_no_change=20, random_state=random_state, batch_size=128
            )
        }
        if CATBOOST_AVAILABLE:
            models['CatBoost'] = CatBoostClassifier(
                iterations=300, depth=6, learning_rate=0.05, random_state=random_state,
                verbose=False, loss_function='MultiClass'
            )
        return models
    
    def train_models(X_train, y_train, X_test, y_test, models=None, cv_folds=5, save_dir="models/saved"):
        """Train models and return results."""
        if models is None:
            models = build_models()
        
        os.makedirs(save_dir, exist_ok=True)
        results = {}
        cv = StratifiedKFold(n_splits=cv_folds, shuffle=True, random_state=42)
        
        print(f"🤖 Training {len(models)} models...")
        
        for name, model in models.items():
            print(f"\n🔁 Training {name}...")
            try:
                model.fit(X_train, y_train)
                y_pred = model.predict(X_test)
                y_proba = model.predict_proba(X_test) if hasattr(model, 'predict_proba') else None
                
                acc = accuracy_score(y_test, y_pred)
                roc_auc = None
                if y_proba is not None:
                    try:
                        if y_proba.shape[1] == 2:
                            roc_auc = roc_auc_score(y_test, y_proba[:, 1])
                        else:
                            roc_auc = roc_auc_score(y_test, y_proba, multi_class='ovr', average='weighted')
                    except:
                        pass
                
                prec = precision_score(y_test, y_pred, average='weighted', zero_division=0)
                rec = recall_score(y_test, y_pred, average='weighted', zero_division=0)
                f1 = f1_score(y_test, y_pred, average='weighted', zero_division=0)
                
                cv_scores = cross_val_score(model, X_train, y_train, cv=cv, scoring='accuracy', n_jobs=-1)
                
                results[name] = {
                    'model': model, 'accuracy': float(acc), 'roc_auc': float(roc_auc) if roc_auc else None,
                    'precision': float(prec), 'recall': float(rec), 'f1': float(f1),
                    'cv_mean': float(cv_scores.mean()), 'cv_std': float(cv_scores.std()),
                    'y_pred': y_pred, 'y_proba': y_proba
                }
                
                print(f"   ✅ Acc={acc:.4f} | ROC-AUC={roc_auc:.4f if roc_auc else 'N/A'} | F1={f1:.4f}")
            except Exception as e:
                print(f"   ❌ {name} failed: {e}")
        
        return results

print(f"✅ Available models: {list(build_models().keys())}")


## 3. Load Feature Data

**In Colab:** Upload your CSV files using the file uploader below, or place them in `/content/Alzhemiers/data/processed/`


In [ ]:
# Colab file upload helper
if IN_COLAB:
    from google.colab import files as colab_files
    print("📤 Upload your feature CSV files (or skip if files already exist):")
    print("   - speech_features.csv")
    print("   - text_features.csv")
    print("   - behavior_features.csv")
    print("   - labels.csv")
    
    uploaded = colab_files.upload()
    if uploaded:
        os.makedirs('data/processed', exist_ok=True)
        for name, content in uploaded.items():
            if name.endswith('.csv'):
                filepath = os.path.join('data/processed', name)
                with open(filepath, 'wb') as f:
                    f.write(content)
                print(f"✅ Saved {name} to {filepath}")
else:
    print("💻 Running locally - using existing files in data/processed/")

# Configuration
DATA_DIR = "data/processed"
os.makedirs(DATA_DIR, exist_ok=True)

SPEECH_CSV = os.path.join(DATA_DIR, "speech_features.csv")
TEXT_CSV = os.path.join(DATA_DIR, "text_features.csv")
BEHAVIOR_CSV = os.path.join(DATA_DIR, "behavior_features.csv")
LABELS_CSV = os.path.join(DATA_DIR, "labels.csv")

# Load data
speech_df = None
text_df = None
behavior_df = None
labels_df = None

print("\n📂 Loading feature data...")

# Try to load speech features
if os.path.exists(SPEECH_CSV):
    speech_df = pd.read_csv(SPEECH_CSV)
    print(f"✅ Loaded speech features: {speech_df.shape}")
    print(f"   Columns: {list(speech_df.columns[:5])}...")
else:
    print(f"⚠️ Speech features not found: {SPEECH_CSV}")
    if IN_COLAB:
        print("   💡 Upload speech_features.csv using the uploader above")
    else:
        print("   Generate using: scripts/speech_input.py")

# Try to load text features
if os.path.exists(TEXT_CSV):
    text_df = pd.read_csv(TEXT_CSV)
    print(f"✅ Loaded text features: {text_df.shape}")
    print(f"   Columns: {list(text_df.columns[:5])}...")
else:
    print(f"⚠️ Text features not found: {TEXT_CSV}")
    if IN_COLAB:
        print("   💡 Upload text_features.csv using the uploader above")
    else:
        print("   Generate using: scripts/text_input.py")

# Try to load behavior features
if os.path.exists(BEHAVIOR_CSV):
    behavior_df = pd.read_csv(BEHAVIOR_CSV)
    print(f"✅ Loaded behavior features: {behavior_df.shape}")
    print(f"   Columns: {list(behavior_df.columns[:5])}...")
else:
    print(f"⚠️ Behavior features not found: {BEHAVIOR_CSV}")
    if IN_COLAB:
        print("   💡 Upload behavior_features.csv using the uploader above")
    else:
        print("   Generate using: scripts/behavior_input.py")

# Load labels
if os.path.exists(LABELS_CSV):
    labels_df = pd.read_csv(LABELS_CSV)
    print(f"✅ Loaded labels: {labels_df.shape}")
    print(f"   Label distribution: {labels_df['label'].value_counts().to_dict() if 'label' in labels_df.columns else 'N/A'}")
else:
    print(f"⚠️ Labels not found: {LABELS_CSV}")
    print("   Expected format: subject_id,label (0=healthy, 1=Alzheimer's)")
    if IN_COLAB:
        print("   💡 Upload labels.csv using the uploader above")


## 4. Prepare Data & Train Speech Models


In [ ]:
# Prepare speech data
speech_results = {}
if speech_df is not None and labels_df is not None:
    # Merge with labels
    id_cols = ['subject_id', 'patient_id', 'id']
    subject_col = next((c for c in id_cols if c in speech_df.columns), None)
    
    if subject_col:
        merged = pd.merge(speech_df, labels_df, on=subject_col, how='inner')
        exclude_cols = id_cols + ['label', 'Label']
        feature_cols = [c for c in merged.columns if c not in exclude_cols]
        
        X = merged[feature_cols].fillna(0).values
        y = merged['label'].values if 'label' in merged.columns else merged['Label'].values
        
        # Split
        X_train, X_test, y_train, y_test = train_test_split(
            X, y, test_size=0.2, random_state=42, stratify=y
        )
        
        # Train
        print(f"\n🤖 Training Speech Models on {X_train.shape[0]} samples...")
        speech_results = train_models(X_train, y_train, X_test, y_test, cv_folds=5)
        
        # Store test labels for fusion
        speech_y_test = y_test
        
        # Find best
        if speech_results:
            best = max(speech_results.items(), key=lambda x: x[1]['accuracy'])
            print(f"\n🏆 Best Speech Model: {best[0]} (Acc={best[1]['accuracy']:.4f})")
    else:
        print("⚠️ No subject ID column found in speech data")
else:
    print("⚠️ Skipping speech: missing data")


## 5. Train Text Models


In [ ]:
# Prepare text data
text_results = {}
if text_df is not None and labels_df is not None:
    id_cols = ['subject_id', 'patient_id', 'id']
    subject_col = next((c for c in id_cols if c in text_df.columns), None)
    
    if subject_col:
        merged = pd.merge(text_df, labels_df, on=subject_col, how='inner')
        exclude_cols = id_cols + ['label', 'Label']
        feature_cols = [c for c in merged.columns if c not in exclude_cols]
        
        X = merged[feature_cols].fillna(0).values
        y = merged['label'].values if 'label' in merged.columns else merged['Label'].values
        
        X_train, X_test, y_train, y_test = train_test_split(
            X, y, test_size=0.2, random_state=42, stratify=y
        )
        
        print(f"\n🤖 Training Text Models on {X_train.shape[0]} samples...")
        text_results = train_models(X_train, y_train, X_test, y_test, cv_folds=5)
        
        # Store test labels for fusion
        text_y_test = y_test
        
        if text_results:
            best = max(text_results.items(), key=lambda x: x[1]['accuracy'])
            print(f"\n🏆 Best Text Model: {best[0]} (Acc={best[1]['accuracy']:.4f})")
    else:
        print("⚠️ No subject ID column found in text data")
else:
    print("⚠️ Skipping text: missing data")


## 6. Train Behavior Models


In [ ]:
# Prepare behavior data
behavior_results = {}
if behavior_df is not None and labels_df is not None:
    id_cols = ['subject_id', 'patient_id', 'id']
    subject_col = next((c for c in id_cols if c in behavior_df.columns), None)
    
    if subject_col:
        merged = pd.merge(behavior_df, labels_df, on=subject_col, how='inner')
        exclude_cols = id_cols + ['label', 'Label']
        feature_cols = [c for c in merged.columns if c not in exclude_cols]
        
        X = merged[feature_cols].fillna(0).values
        y = merged['label'].values if 'label' in merged.columns else merged['Label'].values
        
        X_train, X_test, y_train, y_test = train_test_split(
            X, y, test_size=0.2, random_state=42, stratify=y
        )
        
        print(f"\n🤖 Training Behavior Models on {X_train.shape[0]} samples...")
        behavior_results = train_models(X_train, y_train, X_test, y_test, cv_folds=5)
        
        # Store test labels for fusion
        behavior_y_test = y_test
        
        if behavior_results:
            best = max(behavior_results.items(), key=lambda x: x[1]['accuracy'])
            print(f"\n🏆 Best Behavior Model: {best[0]} (Acc={best[1]['accuracy']:.4f})")
    else:
        print("⚠️ No subject ID column found in behavior data")
else:
    print("⚠️ Skipping behavior: missing data")


## 7. Multimodal Fusion Model

Combine best predictions from all modalities using a meta-learner.


In [ ]:
# Create fusion model
fusion_results = {}
modality_probas = []
modality_labels = None

# Collect best model probabilities from available modalities
if speech_results:
    best_speech_proba = max(speech_results.items(), key=lambda x: x[1]['accuracy'])[1]['y_proba']
    modality_probas.append(('speech', best_speech_proba))
    # Use speech test labels as reference
    if 'speech_y_test' in locals():
        modality_labels = speech_y_test

if text_results:
    best_text_proba = max(text_results.items(), key=lambda x: x[1]['accuracy'])[1]['y_proba']
    modality_probas.append(('text', best_text_proba))
    if modality_labels is None and 'text_y_test' in locals():
        modality_labels = text_y_test

if behavior_results:
    best_behavior_proba = max(behavior_results.items(), key=lambda x: x[1]['accuracy'])[1]['y_proba']
    modality_probas.append(('behavior', best_behavior_proba))
    if modality_labels is None and 'behavior_y_test' in locals():
        modality_labels = behavior_y_test

if len(modality_probas) >= 2 and modality_labels is not None:
    print(f"\n🔗 Creating Fusion Model ({'+'.join([m[0] for m in modality_probas])})...")
    
    # Stack probabilities horizontally
    fusion_X = np.hstack([proba for _, proba in modality_probas])
    
    # Split for meta-learner training
    X_train_fusion, X_test_fusion, y_train_fusion, y_test_fusion = train_test_split(
        fusion_X, modality_labels, test_size=0.2, random_state=42, stratify=modality_labels
    )
    
    # Meta-learner
    meta_learner = LogisticRegression(random_state=42, max_iter=2000, class_weight='balanced')
    meta_learner.fit(X_train_fusion, y_train_fusion)
    
    y_pred_fusion = meta_learner.predict(X_test_fusion)
    y_proba_fusion = meta_learner.predict_proba(X_test_fusion)
    
    acc_fusion = accuracy_score(y_test_fusion, y_pred_fusion)
    roc_auc_fusion = roc_auc_score(y_test_fusion, y_proba_fusion[:, 1]) if y_proba_fusion.shape[1] == 2 else None
    f1_fusion = f1_score(y_test_fusion, y_pred_fusion, average='weighted')
    
    fusion_results = {
        'model': meta_learner,
        'accuracy': acc_fusion,
        'roc_auc': roc_auc_fusion,
        'f1': f1_fusion
    }
    
    print(f"✅ Fusion Model: Acc={acc_fusion:.4f}, ROC-AUC={roc_auc_fusion:.4f if roc_auc_fusion else 'N/A'}, F1={f1_fusion:.4f}")
else:
    print(f"⚠️ Need at least 2 modalities for fusion. Available: {len(modality_probas)}")


## 8. Model Comparison Summary


In [ ]:
# Create comparison table
comparison_data = []

# Add speech models
if speech_results:
    for name, res in speech_results.items():
        comparison_data.append({
            'Modality': 'Speech',
            'Model': name,
            'Accuracy': res['accuracy'],
            'ROC-AUC': res['roc_auc'] if res['roc_auc'] else 0.0,
            'F1-Score': res['f1'],
            'CV Mean': res['cv_mean'],
            'CV Std': res['cv_std']
        })

# Add text models
if text_results:
    for name, res in text_results.items():
        comparison_data.append({
            'Modality': 'Text',
            'Model': name,
            'Accuracy': res['accuracy'],
            'ROC-AUC': res['roc_auc'] if res['roc_auc'] else 0.0,
            'F1-Score': res['f1'],
            'CV Mean': res['cv_mean'],
            'CV Std': res['cv_std']
        })

# Add behavior models
if behavior_results:
    for name, res in behavior_results.items():
        comparison_data.append({
            'Modality': 'Behavior',
            'Model': name,
            'Accuracy': res['accuracy'],
            'ROC-AUC': res['roc_auc'] if res['roc_auc'] else 0.0,
            'F1-Score': res['f1'],
            'CV Mean': res['cv_mean'],
            'CV Std': res['cv_std']
        })

# Add fusion model
if fusion_results:
    comparison_data.append({
        'Modality': 'Fusion',
        'Model': 'Meta-Learner',
        'Accuracy': fusion_results['accuracy'],
        'ROC-AUC': fusion_results['roc_auc'] if fusion_results['roc_auc'] else 0.0,
        'F1-Score': fusion_results['f1'],
        'CV Mean': 0.0,
        'CV Std': 0.0
    })

if comparison_data:
    comparison_df = pd.DataFrame(comparison_data)
    comparison_df = comparison_df.sort_values('Accuracy', ascending=False)
    
    print("\n📊 Model Comparison (sorted by Accuracy):")
    print(comparison_df.to_string(index=False))
    
    # Summary
    print(f"\n✅ Total models trained: {len(comparison_df)}")
    print(f"🏆 Best overall: {comparison_df.iloc[0]['Modality']} - {comparison_df.iloc[0]['Model']} (Acc={comparison_df.iloc[0]['Accuracy']:.4f})")
else:
    print("⚠️ No results to compare")


In [ ]:
# Save models
SAVE_DIR = "models/saved/modality_models"
os.makedirs(SAVE_DIR, exist_ok=True)

timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")

# Save speech models
if speech_results:
    speech_dir = os.path.join(SAVE_DIR, "speech")
    os.makedirs(speech_dir, exist_ok=True)
    for name, res in speech_results.items():
        model_path = os.path.join(speech_dir, f"{name.replace(' ', '_')}_{timestamp}.pkl")
        joblib.dump({
            'model': res['model'],
            'modality': 'speech',
            'accuracy': res['accuracy'],
            'timestamp': timestamp
        }, model_path)
    print(f"✅ Saved {len(speech_results)} speech models to {speech_dir}")

# Save text models
if text_results:
    text_dir = os.path.join(SAVE_DIR, "text")
    os.makedirs(text_dir, exist_ok=True)
    for name, res in text_results.items():
        model_path = os.path.join(text_dir, f"{name.replace(' ', '_')}_{timestamp}.pkl")
        joblib.dump({
            'model': res['model'],
            'modality': 'text',
            'accuracy': res['accuracy'],
            'timestamp': timestamp
        }, model_path)
    print(f"✅ Saved {len(text_results)} text models to {text_dir}")

# Save behavior models
if behavior_results:
    behavior_dir = os.path.join(SAVE_DIR, "behavior")
    os.makedirs(behavior_dir, exist_ok=True)
    for name, res in behavior_results.items():
        model_path = os.path.join(behavior_dir, f"{name.replace(' ', '_')}_{timestamp}.pkl")
        joblib.dump({
            'model': res['model'],
            'modality': 'behavior',
            'accuracy': res['accuracy'],
            'timestamp': timestamp
        }, model_path)
    print(f"✅ Saved {len(behavior_results)} behavior models to {behavior_dir}")

# Save fusion model
if fusion_results:
    fusion_path = os.path.join(SAVE_DIR, f"fusion_meta_learner_{timestamp}.pkl")
    joblib.dump({
        'model': fusion_results['model'],
        'modality': 'fusion',
        'accuracy': fusion_results['accuracy'],
        'timestamp': timestamp
    }, fusion_path)
    print(f"✅ Saved fusion model to {fusion_path}")

print(f"\n💾 All models saved to {SAVE_DIR}")

# In Colab, provide download option
if IN_COLAB:
    print("\n📥 To download models from Colab:")
    print("   1. Use: from google.colab import files; files.download('models/saved/modality_models/...')")
    print("   2. Or mount Google Drive and copy files there")
